In [4]:
# Noise-Cell 1 —— 参数 & 通用工具函数（新 notebook 里先跑这个）

import numpy as np
import os, gc
from scipy.signal import butter, filtfilt, find_peaks

Export_FolderName = "ProcessingTest"

# -------- 和 good 数据保持一致的参数 --------
time_range_1 = 0
time_range_2 = 10000

order         = 3
low_cutoff    = 250
high_cutoff   = 3000

baseline_width      = 5      # 阈值 = baseline_width * rms
peak_gap            = 30
peak_amplitude_limit = 150
peak_width          = 0.003  # 3 ms，对 20kHz 来说大约 60 点

# -------- Intan 读取函数 --------
def load_intan_file(fname):
    """
    自动识别 .rhd / .rhs，返回:
    t:  (T,) 时间轴
    amp: (C, T) 电压数据
    """
    if fname.endswith(".rhd"):
        import importrhdutilities as imp
        result, ok = imp.load_file(fname)
        if not ok:
            raise RuntimeError(f"读取失败: {fname}")
        t = result["t_amplifier"]
        amp = result["amplifier_data"]
    elif fname.endswith(".rhs"):
        import importrhsutilities as imp
        result, ok = imp.load_file(fname)
        if not ok:
            raise RuntimeError(f"读取失败: {fname}")
        t = result["t"]
        amp = result["amplifier_data"]
    else:
        raise ValueError("只支持 .rhd / .rhs")
    return t, amp

# -------- 带通滤波器 --------
def make_bandpass(sampling_rate, low, high, order=3):
    nyq = 0.5 * sampling_rate
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return b, a

# -------- 估计每个通道的 RMS --------
def estimate_rms(data, b, a):
    """
    data: (C, T)
    返回: rms_noise: (C,)
    """
    n_chan = data.shape[0]
    rms = np.zeros(n_chan)
    for i in range(n_chan):
        filt = filtfilt(b, a, data[i])
        rms[i] = np.std(filt)
    return rms

# -------- 从一个通道中抽 spike 片段 --------
def extract_spikes(filtered, thr, peak_gap, limit, peak_width, rate, t):
    """
    filtered: 1D 滤波后数据
    thr: 下阈值
    limit: 上限（artifact 剔除）
    peak_width: 片段总时长（秒）
    rate: 采样率
    t: 时间轴 (T,)

    返回:
      segs: (N, L) 波形片段
      times: (N,) 每个 spike 的时间
    """
    pos, _ = find_peaks(filtered, height=(thr, limit), distance=peak_gap)
    neg, _ = find_peaks(-filtered, height=(thr, limit), distance=peak_gap)
    idx = np.concatenate([pos, neg])
    idx.sort()

    half = int(peak_width / 2 * rate)  # 左右各 half 点
    if half == 0:
        return np.zeros((0, 0)), np.array([])

    # 不能靠得太近边缘
    valid = [x for x in idx if half <= x <= len(filtered) - half]
    if len(valid) == 0:
        return np.zeros((0, 2*half)), np.array([])

    segs = np.zeros((len(valid), 2*half))
    for j, c in enumerate(valid):
        segs[j] = filtered[c-half:c+half]
    times = t[valid]
    return segs, times

In [5]:
# Noise-Cell 2 —— 指定 bad data 的 .rhd 文件列表

data_bad_files = [
    "data_bad/20250715_0610A_WTC_152days_250715_112711.rhd",
    "data_bad/20250715_0610A_WTC_152days_250715_112811.rhd",
    "data_bad/20250715_0610A_WTC_152days_250715_112911.rhd",
    "data_bad/20250715_0610A_WTC_152days_250715_113011.rhd",
    "data_bad/20250715_0610A_WTC_152days_250715_113111.rhd",
]

print("Bad data files:")
for f in data_bad_files:
    print("  ", f)

Bad data files:
   data_bad/20250715_0610A_WTC_152days_250715_112711.rhd
   data_bad/20250715_0610A_WTC_152days_250715_112811.rhd
   data_bad/20250715_0610A_WTC_152days_250715_112911.rhd
   data_bad/20250715_0610A_WTC_152days_250715_113011.rhd
   data_bad/20250715_0610A_WTC_152days_250715_113111.rhd


In [6]:
# Noise-Cell 3 —— 从 bad data 中提取 spike 段，并全部标为噪声(0)

noise_segments_list = []

for fname in data_bad_files:
    print(f"\n[Noise from BAD] Loading {fname} ...")
    t_cur, amp_cur = load_intan_file(fname)

    total_time_bad = t_cur[-1] - t_cur[0]
    sr_bad = int(round(amp_cur.shape[1] / total_time_bad))
    n_chan_bad, n_time_bad = amp_cur.shape
    print(f"  -> sr={sr_bad}, channels={n_chan_bad}, points={n_time_bad}")

    # 1) 裁剪时间范围（和 good 一致）
    start = int(time_range_1 * sr_bad)
    end   = int(min(total_time_bad, time_range_2) * sr_bad)
    t_seg   = t_cur[start:end] - t_cur[start]
    amp_seg = amp_cur[:, start:end]

    # 2) 带通滤波（和 good 一致）
    b_bad, a_bad = make_bandpass(sr_bad, low_cutoff, high_cutoff, order)

    # 3) 每个通道估 RMS，用来设阈值
    rms_bad = estimate_rms(amp_seg, b_bad, a_bad)

    # 4) 抽 spike 段
    for ch in range(n_chan_bad):
        filt = filtfilt(b_bad, a_bad, amp_seg[ch])

        segs, times = extract_spikes(
            filt,
            baseline_width * rms_bad[ch],   # 下阈值
            peak_gap,
            peak_amplitude_limit,
            peak_width,
            sr_bad,
            t_seg
        )
        if segs.shape[0] == 0:
            continue

        noise_segments_list.append(segs)
        print(f"  ch {ch:03d}: 捕获噪声样本 {segs.shape[0]} 个")

    del t_cur, amp_cur, t_seg, amp_seg
    gc.collect()

# 5) 聚合所有 bad-data 噪声
if len(noise_segments_list) > 0:
    # 注意：这里假设 peak_width=0.003、采样率 ≈ 20kHz，所以长度应该是 60
    X_noise_bad = np.concatenate(noise_segments_list, axis=0)  # (N_bad, L)
else:
    X_noise_bad = np.empty((0, 60))

print("\n[Noise from BAD] 总噪声样本数量:", X_noise_bad.shape[0])

# 6) 保存
os.makedirs(Export_FolderName, exist_ok=True)
np.savez_compressed(
    os.path.join(Export_FolderName, "bad_noise_spikes.npz"),
    X=X_noise_bad
)
print("已保存 bad 噪声样本到:", os.path.join(Export_FolderName, "bad_noise_spikes.npz"))


[Noise from BAD] Loading data_bad/20250715_0610A_WTC_152days_250715_112711.rhd ...

Reading Intan Technologies RHD Data File, Version 3.3

Found 240 amplifier channels.
Found 15 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 0 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

File contains 60.000 seconds of data.  Amplifiers were sampled at 20.00 kS/s.

Allocating memory for data...
Reading data from file...
10% done...
20% done...
30% done...
40% done...
50% done...
60% done...
70% done...
80% done...
90% done...
Parsing data...
No missing timestamps in data.
Done!  Elapsed time: 4.6 seconds
  -> sr=20000, channels=240, points=1200000
  ch 000: 捕获噪声样本 10 个
  ch 001: 捕获噪声样本 7 个
  ch 002: 捕获噪声样本 14 个
  ch 003: 捕获噪声样本 5 个
  ch 004: 捕获噪声样本 8 个
  ch 005: 捕获噪声样本 97 个
  ch 006: 捕获噪声样本 8 个
  ch 007: 捕获噪声样本 1 个
  ch 008: 捕获噪声样本 10 个
  ch 009: 捕获噪声样本 21 个
  ch 010: 捕获噪声样本 59 个
  ch 011: 捕获噪声

In [9]:
# Cell A0 — 加载人工标注的 spike 片段 (good data)，允许轻微长度不一致，用 min 截断

import numpy as np
import h5py
import os
from collections import Counter

Export_FolderName = "ProcessingTest"

# ========= 1) 读取 segments =========
seg_path = os.path.join(Export_FolderName, "all_peak_segments_marking.h5")
with h5py.File(seg_path, "r") as hdf:
    # 兼容不同 group 名：优先用标注版，如果没有就退到原始名
    if "all_peak_segments_marking" in hdf:
        grp = hdf["all_peak_segments_marking"]
    else:
        grp = hdf["all_peak_segments"]
    all_segments = []
    # 假定通道 key 是 '0','1','2',...
    # 按照 key 排序，保证通道顺序一致
    keys = sorted(grp.keys(), key=lambda x: int(x))
    for key in keys:
        dset = grp[key]
        ch_segments = [np.array(seg) for seg in dset]   # 每个 seg 是 (60,)
        all_segments.append(ch_segments)

n_ch_seg = len(all_segments)
print(f"segments 通道数: {n_ch_seg}")

# ========= 2) 读取 labels =========
lab_path = os.path.join(Export_FolderName, "all_clusters_marking.h5")
with h5py.File(lab_path, "r") as hdf:
    if "all_clusters_marking" in hdf:
        dset_lab = hdf["all_clusters_marking"]
    else:
        # 如果将来改名，这里可以适配
        dset_lab = list(hdf.values())[0]
    all_labels = [np.array(dset_lab[i]) for i in range(len(dset_lab))]

n_ch_lab = len(all_labels)
print(f"labels 通道数:   {n_ch_lab}")

# ========= 3)（可选）读 datapoint，一起检查 =========
dp_path = os.path.join(Export_FolderName, "all_peak_datapoint_marking.h5")
all_dps = None
if os.path.exists(dp_path):
    with h5py.File(dp_path, "r") as hdf:
        if "all_peak_datapoint_marking" in hdf:
            dset_dp = hdf["all_peak_datapoint_marking"]
        else:
            dset_dp = list(hdf.values())[0]
        all_dps = [np.array(dset_dp[i]) for i in range(len(dset_dp))]
    print(f"datapoint 通道数: {len(all_dps)}")
else:
    print("⚠️ 未找到 all_peak_datapoint_marking.h5，先只用 segments + labels。")

# ========= 4) 汇总到 X_full, y_full =========

X_full_list, y_full_list = [], []
bad_channels = []

n_channels = min(n_ch_seg, n_ch_lab)  # 防止一边多一边少

for ch in range(n_channels):
    segs_ch = all_segments[ch]   # list of (L_seg,) arrays
    labs_ch = all_labels[ch]     # 1D array
    n_seg = len(segs_ch)
    n_lab = len(labs_ch)
    n_dp  = len(all_dps[ch]) if all_dps is not None else None

    if n_seg == 0 or n_lab == 0:
        bad_channels.append(ch)
        continue

    if n_dp is not None and (n_seg != n_lab or n_lab != n_dp):
        print(f"通道 {ch:03d} 长度不一致: seg={n_seg}, lab={n_lab}, dp={n_dp} -> 用 min 截断")
    elif n_seg != n_lab:
        print(f"通道 {ch:03d} 长度不一致: seg={n_seg}, lab={n_lab} -> 用 min 截断")

    L = min(n_seg, n_lab)
    if L == 0:
        bad_channels.append(ch)
        continue

    # 截断对齐
    X_full_list.extend(segs_ch[:L])
    y_full_list.extend(labs_ch[:L])

X_full = np.array(X_full_list)
y_full  = np.array(y_full_list)

print("\n===== 汇总结果 =====")
print("X_full shape =", X_full.shape)
print("y_full shape =", y_full.shape)
print("Label distribution:", Counter(y_full.tolist()))
print("被完全忽略的通道数:", len(bad_channels), " ->", bad_channels[:20], "...")

segments 通道数: 240
labels 通道数:   240
datapoint 通道数: 240

===== 汇总结果 =====
X_full shape = (77172, 60)
y_full shape = (77172,)
Label distribution: Counter({2: 56675, 1: 17228, 0: 2336, 100: 933})
被完全忽略的通道数: 6  -> [79, 90, 94, 168, 213, 214] ...


In [10]:
# Cell A3' — 构造新的 Task A 数据集（加入 bad 噪声）

import numpy as np, os
from collections import Counter

print("Good 标注数据形状:", X_full.shape, "标签分布:", Counter(y_full.tolist()))

# 1) 加载 bad 噪声波形
noise_npz_path = os.path.join(Export_FolderName, "bad_noise_spikes.npz")
noise_npz = np.load(noise_npz_path)
X_noise_bad = noise_npz["X"]       # (N_bad, L)
print("Bad 噪声数据形状:", X_noise_bad.shape)

# 检查长度是否一致（都应该是 60）
assert X_noise_bad.shape[1] == X_full.shape[1], "bad 噪声长度和 good 数据不一致，需要对齐后再用"

# 2) 先从 good 标注里选出 Task A 用到的部分
#    标签含义：
#       0   = 噪声
#       1   = 皮层
#       2   = 脊髓
#       100 = 不确定
#    这里的策略：
#       0 和 100 都当成「噪声」
#       1 和 2 当成「信号」

mask_valid = np.isin(y_full, [0, 1, 2, 100])
X_good = X_full[mask_valid]
y_good_raw = y_full[mask_valid]

# 映射为 Task A 标签：0=噪声, 1=信号
yA_good = np.zeros_like(y_good_raw, dtype=np.int64)
yA_good[np.isin(y_good_raw, [1, 2])] = 1          # 皮层/脊髓 -> 信号
yA_good[np.isin(y_good_raw, [0, 100])] = 0        # 噪声 + 不确定 -> 噪声

print("Task A (仅 good 数据) 标签分布:", Counter(yA_good.tolist()))

# 3) 为 bad 噪声构造标签，全是 0（噪声）
yA_bad = np.zeros(X_noise_bad.shape[0], dtype=np.int64)

# 4) 合并 good + bad
X_A_all = np.concatenate([X_good, X_noise_bad], axis=0)
y_A_all = np.concatenate([yA_good, yA_bad], axis=0)

print("Task A 合并后形状:", X_A_all.shape, "标签分布:", Counter(y_A_all.tolist()))

# 5) Stratified 划分 train / val / test
from sklearn.model_selection import train_test_split

def stratified_split_indices(y, train=0.7, val=0.15, test=0.15, random_state=42):
    idx = np.arange(len(y))
    tr_idx, tmp_idx, y_tr, y_tmp = train_test_split(
        idx, y, train_size=train, stratify=y, random_state=random_state
    )
    val_ratio = val / (val + test)
    val_idx, te_idx, _, _ = train_test_split(
        tmp_idx, y_tmp, train_size=val_ratio, stratify=y_tmp, random_state=random_state
    )
    return tr_idx, val_idx, te_idx

trA2, vaA2, teA2 = stratified_split_indices(y_A_all, 0.7, 0.15, 0.15, random_state=42)

print("Task A v2 训练/验证/测试尺寸:",
      len(trA2), len(vaA2), len(teA2))

# 6) 保存新的 Task A 数据集
out_npz = os.path.join(Export_FolderName, "taskA_with_badNoise.npz")
np.savez_compressed(
    out_npz,
    X=X_A_all, y=y_A_all,
    tr=trA2, val=vaA2, te=teA2
)

print("Task A v2 已保存到:", out_npz)

Good 标注数据形状: (77172, 60) 标签分布: Counter({2: 56675, 1: 17228, 0: 2336, 100: 933})
Bad 噪声数据形状: (23406, 60)
Task A (仅 good 数据) 标签分布: Counter({1: 73903, 0: 3269})
Task A 合并后形状: (100578, 60) 标签分布: Counter({1: 73903, 0: 26675})
Task A v2 训练/验证/测试尺寸: 70404 15087 15087
Task A v2 已保存到: ProcessingTest/taskA_with_badNoise.npz
